# 23. SQL Magic in Jupyter

*Module: 6 — Python access · Companion summary: `lectures/23-sql-magic.md` · SQL engine: SQLite (in-memory)*

## What you will study

- know what SQL Magic is (line/cell magics from ipython-sql)
- try the magics if available, with a runnable DB-API fallback either way
- map magic syntax ↔ plain Python calls

---
## Setup — build the sample database this notebook runs on
*Run the cell below first: it creates an in-memory SQLite database and the `q()` helper that prints every query's result set.*

## 📗 SQL Magic (ipython-sql)

In the course's labs, the first cell does:
```bash
pip install ipython-sql           # (commented here — see below)
%load_ext sql
%sql sqlite://                    # connect (in-memory) — in labs: %sql ibm_db... + credentials
```
Then SQL reads like a natural language: `%sql SELECT ...` (one line) or
```
%%sql
SELECT ...
```
(a whole cell, no Python). If the optional package isn't installed in this
environment, the magics below simply print a hint and the DB-API equivalent still
runs — nothing fails.

In [ ]:
# Optional: uncomment to try the real magics
# !pip install ipython-sql

try:
    ipython = get_ipython()
    ipython.run_line_magic('load_ext', 'sql')
    ipython.run_line_magic('sql', 'sqlite://')
    print('✅ SQL Magic loaded - the %%sql cells below work')
except Exception as e:
    print('ℹ️ SQL Magic unavailable here:', type(e).__name__, '-')
    print('   using plain sqlite3 (DB-API) for every query in this notebook: same SQL.')

## 🧪 Try a line magic (`%sql`) — or the equivalent DB-API call

In [ ]:
import sqlite3           # ensure the DB-API import exists even without the magics
try:
    ipython = get_ipython()
    out = ipython.run_line_magic('sql', 'SELECT 1 AS one')
    print(out)
except Exception:
    print('%sql equivalent in pure Python:')
    print(sqlite3.connect(':memory:').execute('SELECT 1 AS one').fetchall())

## 🧪 A second magic for flavor: `%pwd` (built into IPython, no install needed)

In [ ]:
print(get_ipython().run_line_magic('pwd', ''))

## 🧪 A %%sql-style cell, done in DB-API

In [ ]:
import sqlite3
import textwrap

conn = sqlite3.connect(':memory:')  # throw-away in-memory database
conn.execute("PRAGMA foreign_keys = ON")

def q(sql):
    """Run a query, print formatted SQL, and display the result set as a table."""
    clean_sql = textwrap.dedent(sql).strip()

    # 1. Print formatted SQL query
    print("\n" + "=" * 60)
    print("QUERY:")
    print(textwrap.indent(clean_sql, "  "))
    print("-" * 60)

    cur = conn.execute(sql)
    rows = cur.fetchall()

    if cur.description:  # SELECT / statement returning rows
        headers = [d[0] for d in cur.description]
        str_rows = [[("" if v is None else str(v)) for v in r] for r in rows]

        # Calculate dynamic column widths for clean alignment
        widths = [
            max(len(h), *(len(r[i]) for r in str_rows), 1)
            for i, h in enumerate(headers)
        ] if str_rows else [len(h) for h in headers]

        header_line = " | ".join(h.ljust(widths[i]) for i, h in enumerate(headers))
        sep_line = "-+-".join("-" * widths[i] for i in range(len(headers)))

        print(header_line)
        print(sep_line)
        for r in str_rows:
            print(" | ".join(val.ljust(widths[i]) for i, val in enumerate(r)))
        if not str_rows:
            print("(0 rows returned)")
    else:  # DDL / DML (CREATE, INSERT, UPDATE, DELETE)
        affected = cur.rowcount
        print(f"Done (rows affected: {affected})" if affected != -1 else "Done.")

    print("=" * 60)

print('Connection ready:', conn)

q("""
SELECT 'magic-free sql' AS note, 2 * 21 AS the_answer
""")

## 💡 Tips & tricks

- Magics are *sugar* over the same DB-API underneath — SQL identical, ergonomics differ.
- `%sql` handles credentials in one place (labs: Cloud DB2 credential string) — that's
  its main real-world advantage in this course.
- If your `%sql` connection is to a **file**: `%%sql sqlite:///path/to/db.sqlite`.

## 🎯 Your turn

1. Write a %%sql-style multi-line query result in DB-API (any SELECT).
2. Try `get_ipython().run_line_magic('lsmagic', '')` and count line magics (just for fun).

In [ ]:
# 💡 Solution
conn = sqlite3.connect(':memory:')
conn.execute("CREATE TABLE t (a, b)")
conn.execute("INSERT INTO t VALUES (1, 'x')")
cur = conn.execute("""
SELECT a, b FROM t WHERE b = 'x'
""")
print(cur.fetchall())

try:
    mags = get_ipython().run_line_magic('lsmagic', '').magics_manager.magics
    print('line magics available:', len(mags.get('line', {})))
except Exception as e:
    print('lsmagic introspection unavailable:', e)